In [ ]:
# ============================================================
# SILVER LAYER - FACT VISIT PIPELINE (STREAMING + UPSERT)
# ============================================================
# Purpose:
# - Ingest streaming visit data (Bronze layer)
# - Enrich with static dimension tables (patient, hospital, diagnosis)
# - Apply basic cleaning and transformations
# - Load into Silver fact table (Delta) using upsert (MERGE)
#
# Architecture Pattern:
# - Stream (fact) + Batch (dimensions) -> correct
# - Structured Streaming (foreachBatch)
# - Delta Lake MERGE for incremental updates
# - Checkpointing for fault tolerance
#
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql.functions import col, to_date, current_timestamp

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
check_point = "Files/healthcare_data/checkpoint/silver_visit"

visit_bronze_table = "DE_Learning_LH.visit_bronze.visit"
patient_bronze_table = "DE_Learning_LH.patient_bronze.patient"
hospital_bronze_table = "DE_Learning_LH.hospital_bronze.hospital"
diagnosis_bronze_table = "DE_Learning_LH.diagnosis_bronze.diagnosis"

# NOTE: Make sure this is defined in your environment
# Example:
# visit_silver_table = "DE_Learning_LH.silver.fact_visit"

# ------------------------------------------------------------
# Step 1: Read streaming fact data (Bronze)
# ------------------------------------------------------------
# - Streaming ingestion ensures incremental processing
# - Drop duplicates based on business key (visit_id)
df_visit_bronze = (
    spark.readStream.table(visit_bronze_table)
    .dropDuplicates(["visit_id"])
)

# ------------------------------------------------------------
# Step 2: Read dimension tables (STATIC - batch mode)
# ------------------------------------------------------------
# IMPORTANT:
# - Dimensions are read as batch (NOT streaming)
# - Avoids stream-stream join limitations
# - Ensures stable and deterministic joins
df_patient = (
    spark.read.table(patient_bronze_table)
    .dropDuplicates(["patient_id"])
)

df_hospital = (
    spark.read.table(hospital_bronze_table)
    .dropDuplicates(["hospital_id"])
)

df_diagnosis = (
    spark.read.table(diagnosis_bronze_table)
    .dropDuplicates(["diagnosis_code"])
)

# ------------------------------------------------------------
# Step 3: Rename columns to avoid ambiguity
# ------------------------------------------------------------
# Avoid collisions during join (e.g. multiple 'city' columns)
df_patient = df_patient.withColumnRenamed("city", "patient_city")
df_hospital = df_hospital.withColumnRenamed("city", "hospital_city")

# ------------------------------------------------------------
# Step 4: Prepare aliases for clean joins
# ------------------------------------------------------------
v = df_visit_bronze.alias("v")
p = df_patient.alias("p")
h = df_hospital.alias("h")
d = df_diagnosis.alias("d")

# ------------------------------------------------------------
# Step 5: Join fact with dimension tables
# ------------------------------------------------------------
# Design:
# - LEFT joins preserve all visit records
# - Dimensions enrich data
# - Using explicit select avoids duplicate columns
df_combined = (
    v
    .join(p, col("v.patient_id") == col("p.patient_id"), "left")
    .join(h, col("v.hospital_id") == col("h.hospital_id"), "left")
    .join(d, col("v.diagnosis_code") == col("d.diagnosis_code"), "left")
    .select(
        col("v.visit_id"),
        col("v.patient_id"),
        col("v.diagnosis_code"),
        to_date(col("v.admission_date")).alias("admission_date"),
        to_date(col("v.discharge_date")).alias("discharge_date"),
        col("h.bed_count"),
        col("v.cost")
    )
    # Add processing/audit timestamp
    .withColumn("process_ts", current_timestamp())
)

# ------------------------------------------------------------
# Step 6: Define MERGE logic for Silver fact table
# ------------------------------------------------------------
def merge_fact_visit(batch_df, batch_id):
    """
    Perform incremental upsert (MERGE) into Silver fact table.

    Logic:
    - If table does not exist → initialize it (first load)
    - Otherwise:
        - Match on visit_id (business key)
        - Update existing records
        - Insert new records

    Ensures:
    - Idempotent processing
    - No duplicates
    - Supports late-arriving data
    """

    if not spark.catalog.tableExists(visit_silver_table):
        # Initial table creation
        batch_df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(visit_silver_table)
        return

    fact = DeltaTable.forName(spark, visit_silver_table)

    (
        fact.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.visit_id = s.visit_id"
        )
        .whenMatchedUpdateAll()       # Update existing records
        .whenNotMatchedInsertAll()    # Insert new records
        .execute()
    )

# ------------------------------------------------------------
# Step 7: Start streaming job (incremental execution)
# ------------------------------------------------------------
(
    df_combined
    .writeStream
    .foreachBatch(merge_fact_visit)          # Custom batch logic
    .trigger(availableNow=True)              # Process available data and stop
    .option("checkpointLocation", check_point)  # Required for recovery
    .start()
)